## Week 10 and 11 Assignment - DATASCI200 Introduction to Data Science Programming, UC Berkeley MIDS

Write code in this Jupyter Notebook to solve the following problems. Please upload this **Notebook** with your solutions to gradescope. 

Assignment due date: 11:59PM PT the night before the Week 12 Live Session. Do **NOT** push/upload the data file. 

## Objectives

* Analyze and glean insights from a real dataset using pandas
* Apply pandas for exploratory analysis, information gathering, and discovery
* Demonstrate cleaning data and answering questions

## General Guidelines:

- This is a **real** dataset and so it may contain errors and other pecularities to work through
- This dataset is ~218mb, which will take some time to load (and probably won't load in Google Sheets or Excel)
- If you make assumptions, annotate them in your responses
- While there is one code/markdown cell positioned after each question as a placeholder, some of your code/responses may require multiple cells
- Double-click the markdown cells that say for example **1a answer here:** to enter your written answers. If you need more cells for your written answers, make them markdown cells (rather than code cells)
- This homework assignment is not autograded because of the variety of responses one could give. 
  - Please upload this notebook to the autograder page and the TAs will manually grade it. 
  - Ensure that each cell is run and outputs your answer for ease of grading! 
  - Highly suggest to do a `restart & run all` before uploading your code to ensure everything runs and outputs correctly.
  - Answers without code (or code that runs) will be given 0 points.
- **This is meant to simulate real world data so you will have to do some external research to determine what some of the answers are!** 

## Dataset

You are to analyze campaign contributions to the 2016 U.S. presidential primary races made in California. Use the csv file located here: https://drive.google.com/file/d/1Lgg-PwXQ6TQLDowd6XyBxZw5g1NGWPjB/view?usp=sharing. You should download and save this file in the same folder as this notebook is stored.  This file originally came from the U.S. Federal Election Commission (https://www.fec.gov/).

**DO NOT PUSH THIS FILE TO YOUR GITHUB REPO!**

- Best practice is to not have DATA files in your code repo. As shown below, the default load is outside of the folder this notebook is in. If you change the folder where the file is stored please update the first cell!
- If you do accidentally push the file to your github repo - follow the directions here to fix it: https://docs.google.com/document/d/15Irgb5V5G7pKPWgAerH7FPMpKeQRunbNflaW-hR2hTA/edit?usp=sharing

Documentation for this data can be found here: https://drive.google.com/file/d/11o_SByceenv0NgNMstM-dxC1jL7I9fHL/view?usp=sharing

## Data Questions

You are working for a California state-wide election campaign. Your boss wants you to examine historic 2016 election contribution data to see what zipcodes are more supportive of fundraising for your candidate. 

Your boss asks you to filter out some of the records:
- Only use primary 2016 contribution data (more like how your race is).
- Concentrate on Bernie Sanders as a candidate (most a like your candidate)

The questions your boss wants answered is:
- Which zipcode (5-digit zipcode) had the highest count of contributions and the most dollar amount?
- What day(s) of the month do most people donate?

## Setup

Run the cell below as it will load the data into a pandas dataframe named `contrib`. Note that a custom date parser is defined to speed up loading. If Python were to guess the date format, it would take even longer to load.

In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime

# These commands below set some options for pandas and to have matplotlib show the charts in the notebook
pd.set_option('display.max_rows', 1000)
pd.options.display.float_format = '{:,.2f}'.format

# Define a date parser to pass to read_csv
d = lambda x: datetime.strptime(x, '%d-%b-%y')

# Load the data
# We have this defaulted to the folder OUTSIDE of your repo - please change it as needed
contrib = pd.read_csv('/Users/lacey/github/mids-datasci200-summer2026-lacey-payne/P00000001-CA.csv', index_col=False, parse_dates=['contb_receipt_dt'], date_format=d)
# Since I have pandas 3.0.3, I replaced "date_parser" with "date_format" to get rid of:"TypeError: read_csv() got an unexpected keyword argument 'date_parser'"
# To figure out how to solve the error since it was different from the mixed types warning mentioned, I ran "pd.show_versions()" as Line 2 to see what version of pandas I had, then deleted the line
# I then searched "what is the keyword argument to use for 'date_parser" in pandas 3.0.3", the fifth result was for a Github page, and the summary shown had the needed information

# Note - for now, it is okay to ignore the warning about mixed types. 

***
## 1. Initial Data Checks (50 points)

First we will take a preliminary look at the data to check that it was loaded correctly and contains the info we need.

The questions to answer at the end of this section:
- Do we have the correct # of columns and rows. 
- Do the records contain data for the questions we want to answer 
- What columns are important? 
- What columns can be dropped?
- What are the data problems?

**1a.** Print the *shape* of the data. Does this match the expectation? (2 points)

In [ ]:
# 1a YOUR CODE HERE
data = contrib
shape = data.shape
print("Shape = {}" .format(shape))


Yes, this matches the expectation; CA is a populous state, so the number of voters (rows) would reasonably be expected to be large. There is a finite amount of specific, trackable data points that can or should be collected for posterity. Understanding the questions that need to be answered, the number of rows also appears reasonable.

# **1a answer here:** 
Shape = (1125659, 18)
The .csv file has 1125659 total rows and 18 columns

**1b.** Print a list of column names. Are all the columns included that are in the documentation? (2 points)

In [ ]:
# 1b YOUR CODE HERE
pd.set_option('display.max_columns', None)
data.head()
print(data.columns.tolist())


# **1b answer here:**
Yes, all the columns that are in the documentation are included. I verified this by first returning to 1a., looking at the shape of the data, confirming the number of columns printed above matched the number listed in the shape, and inferring from that that all columns from the documentation must be included. My initial code included only lines 2 & 3; I added line 4 to run print(data.columns.tolist()) and double-check that the actual names were the same rather than only the number of columns.

**1c** Print out the first five rows of the dataset. How do the columns `cand_id`, `cand_nm` and `contbr_st` look? (3 points)

In [ ]:
# 1c YOUR CODE HERE
print (data[['cand_id', 'cand_nm', 'contbr_st']].dtypes)
data.head()

# **1c answer here:** 
cand_id, cand_nm, and contbr_st are all strings. Each entry in the cand_id column has 9 total characters, beginning with the letter 'P' and followed by 8 numbers. Each of the two strings listed in the cand_nm column is matched with its own specific value in the cand_id column, showing that each candidate has their own candidate id number. All entries in the contbr_state column are "CA". I postulate that cand_id is used alongside cand_nm as a way for anonymization if necessary, as it appears to be an internal metric and the cand_nm column could be hidden if necessary, and as a data verification measure (the logic being that it would be more likely for one column to contain errors/mispellings than both of them), and that the contbr_st being "CA" for the first five columns makes sense and will likely continue for a majority if not all of the column, as the data is from a previous primary race in California. 

**1d.** Print out the values for the column `election_tp`. In your own words, based on the documentation, what information does the `election_tp` variable contain? Do the values in the column match the documentation? (3 points)

In [ ]:
# 1d YOUR CODE HERE
pd.set_option('display.max_rows', None)
print(data['election_tp'])

# **1d answer here:** 
The 'election_tp' variable shows the election type - "P" for primary, and "2016" for year indicates the election type was the primary election that occurred in 2016. Yes, the values match the documentation; printing a Series prints the index and variables without adding a header row to the row count.

**1e.** Print out the datatypes for all of the columns. What are the datatypes for the `contbr_zip`, `contb_receipt_amt`, `contb_receipt_dt`? (5 points)

In [ ]:
#1e YOUR CODE HERE
print(data.dtypes)


# **1e answer here:** 
contbr_zip is an object, contb_receipt_amt is a float, and contb_receipt_dt is a string

**1f.** What columns have the most non-nulls?  Would you recommend to drop any columns based on the number of nulls? (5 points)

In [ ]:
# 1f YOUR CODE HERE 
non_null_counts = data.count()
sorted_non_null_counts = non_null_counts.sort_values(ascending=True)
print(sorted_non_null_counts)

# **1f answer here:** 
tran_id, file_num, cmte_id, form_tp, contb_receipt_amt, contb_receipt_dt, contbr_nm, cand_nm, cand_id, and contbr_st have the most non-nulls. receipt_desc, memo_cd, memo_txt, and cont_employer have the most nulls, as each is in the six figures vs the remaining columns having seven figures and being at or near the total number of non-header rows. I would argue that each of these could be deleted not due to the higher number of nulls, but because they have nothing to do with the questions that need answering (which zip code had the highest count of contributions and the most dollar amount + which days of the month do most people donate, concentrating on candidate Bernie Sanders). If the data were for a currently ongoing election, I would advocate for keeping the memo_text column to ensure that if a donor wants their contribution allocated a certain way the election campaign could honor that request, but since this is all historical data, there's no need for it. 

**1g.** A column we know that we want to use is the cand_nm column.  From the documentation each candidate is a unique candidate id also. Check data quality of `cand_id` column to see if it matches `cand_nm` column. Specifically check to ensure our targetted candidate 'Bernard Sanders' always has the same cand_id throughout. Any issues with `cand_nm` matching `cand_id`? (5 points)

In [ ]:
# 1g YOUR CODE HERE
data.groupby('cand_id')['cand_nm'].nunique()


**1g answer here:** 
cand_nm matches cand_id throughout; each candidate id number matches exactly one name.

**1h.** Another area to check is to make sure all of the records are from California. Check the `contbr_st` column - are there any records outside of California based on `contbr_st`? (5 points)

In [ ]:
# 1h YOUR CODE HERE
data['contbr_st'].value_counts()

- **1h answer here:** 

There are no records outside of CA in the 'contbr_st' column. 1125669 is the total number of rows, and CA is the listed value for 'contbr_st' in all rows. Plus, CA is the only state listed; if there were other values in the 'contbr_st' column, their names would have populated as well.

**1i.** The next column to check for the analysis is the `tran_id` column. This column could be the primary key so look for duplicates. How many duplicate entries are there? Any pattern for why are there duplicate entries? (5 points)

In [ ]:
# 1i YOUR CODE HERE
print(data['tran_id'].duplicated().sum())
pd.set_option('display.max_rows', 20)
data[data['tran_id'].duplicated(keep=False)].sort_values('tran_id')


There are 3,454 duplicated tran_id entries. Looking at rows 386, 835, and 532 (contbr_nm "PARKS, PATRICIA MRS.", receipt amount 200.00 and receipt date 28-NOV-15,), all three of their file_num entries are different (1057796, 1057799, and 1057798). The two duplicate tran_id entries on rows 428 and 820 also have different file_num columns; 1057796 and 1057799, which are the same file_num for rows 386 & 835, respectively. The other columns displayed as duplicates in the selection of rows I pulled to view the duplicate entries and check for patterns appear to be refunds, and are not true duplicates; the tran_id columns match, but the other data does not. Additionally, the refunds each have either NaN or G2016 for election_tp, so these would not be appliable to the question we're trying to answer.

**1j.** Another column to check is the `contb_receipt_amt` that shows the donation amounts. How many negative donations are included? What do negative donations mean? Please show at least pull a few rows to look at the records with negative donations. Do these records match with the expectation of why a negative donation would happen? (5 points)

In [ ]:
# 1j YOUR CODE HERE
len(data[data['contb_receipt_amt'] < 0])
data[data['contb_receipt_amt'] < 0]

**1k.** One more column to look at is the date of donation column. Are there any dates outside of the primary period (defined as 1 Jan 2014 to 7 June 2016)? Are the dates well-formatted for our analysis? (5 points)

In [ ]:
# 1k YOUR CODE HERE
print(data['contb_receipt_dt'].min())
print(data['contb_receipt_dt'].max())
data['contb_receipt_dt'].dtype
data['contb_receipt_dt'] = pd.to_datetime(data['contb_receipt_dt'], format='%d-%b-%y')
data['contb_receipt_dt'].dtype
print(data['contb_receipt_dt'].min())
print(data['contb_receipt_dt'].max())
outside_range = data[(data['contb_receipt_dt'] < '2014-01-01') | (data['contb_receipt_dt'] > '2016-06-07')]
print(outside_range.shape[0])   # number of rows outside the primary period
(outside_range)            # rows outside the primary period

**1k answer here:**
Upon initial review, it appeared no dates were outside of the primary period. The data in its current state is not well-formatted for our analysis. The StringDtype is not a real date, it's an object/text. In the setup cell, a lambda function was used to define a date parser to read the .csv file but it didn't actually convert the cells from strings to datetime. To perform further analysis, we will need a format string rather than a parser function. Using line 5 in 1k converts contb_receipt_dt to datetime, ensuring the type is correct for the entire column.   
After converting the data to datetime and running min and max again, it appears there are some dates outside of the primary period. Since we will not need these for our analysis, it is sensible to determine how many there are now. There are 451,372 rows outside of the primary period

**1l.** Finally, answer the initial questions in the cells below (5 points)

**1l.1** Do we have the correct # of columns and rows.
No, we will need to eliminate any rows for duplicate donations and/or ones occuring on dates outside of the primary period. But currently, we have a total of 1125659 rows and 18 columns


**1l.2** Do the records contain data for the questions we want to answer?
Yes. We want to determine which zipcode had the highest count of contributions and most dollar amount, and on which day(s) of the month most people donated during the 2016 primary election for candidate Bernie Sanders. Since the question asks which day of the month rather than which week day, no additional features will need to be added to the contb_receipt_dt column to enable the analysis to occur.

**1l.3** What columns are important?

cand_id, cand_nm, contrb_zip, contb_receipt_amt, contb_receipt_dt, election_tp, tran_id

**1l.4** What columns can be dropped?
cmte_id, contbr_nm, contbr_city, contbr_st, contbr_employer, contbr_occupation, receipt_desc, memo_cd, memo_text, form_tp, file_num

**1l.5** What are the data problems?
there are duplicate records, records from outside of the primary period, negative donations (receipts or reallocations), the contb_receipt_dt column was initially a string instead of datetime, from reviewing the records to answer other questions, it appears that contbr_zip is an object rather than int, and it has been formatted numerous different ways on the spreadsheet (dollar amounts, appropraite 5-digit zip code, zip + 4). 

**1l.6** List any assumptions so far:
- Once the data quality is fully fixed (specifically, the contbr_zip column now that contb_receipt_dt column is in datetime) and filtered to exclude donations outside of the primary period, duplicate transactions, and negative transactions, we will have everything we need to complete analysis and answer the two questions posed.
- The dataset is complete and includes all donations
- The data is accurate

***
## 2. Data filtering and data quality fixes (30 points)

Now that we have a basic understanding of the data, let's filter out the records we don't need and fix the data. Do each of the following problems sequentially so that at the end the dataframe is filtered and cleaned for the analysis. (that is, use the dataframe answer for 2a to start 2b etc)

**2a.** From the dataset filter out (remove) any election_tp not in the primary election for 2016. Also filter for the primary dates (defined as 1 Jan 2014 to 7 June 2016). Print/show the shape of the dataframe after the filtering is complete. (5 points)

**1j answer here:**
There are 11,896 negative contributions in the dataset. The negative contributions are likely refunds (as shown in 1i), charge backs, redesigntaion to general (an excess contribution made for the primary applied to the general election instead), or redesignation to spouse (so the donor directed the campaign to treat a donation meant for one election (primary) as if it were for another election (general) involving the same candidate)

In [ ]:
# 2a YOUR CODE HERE
df = data[(data["election_tp"] == "P2016") & (data["contb_receipt_dt"] >= "2014-01-01") & (data["contb_receipt_dt"] <= "2016-06-07")] 
print("Shape = {}".format(df.shape)) #This is the shape of the dataframe after filtering for the primary period and election type

**2b.** From the dataset filter out (remove) any candidate that is not Bernie Sanders. Print/show the shape of the dataframe after the filtering is complete. (5 points)

In [ ]:
# 2b YOUR CODE HERE
df = df[df["cand_nm"] == "Sanders, Bernard"]
print("Shape = {}".format(df.shape)) #This is the shape of the dataframe after filtering for the primary period, election type, and candidate ID

**2c.** The `contbr_zip` column is not formatted well for our analysis. Make a new zipcode column that is the five-digit zipcodes. Filter out any records outside of California based on the zipcode. Print/show the shape of the dataframe after the filtering is complete. (10 points).

- You will have to research what the valid 5-digit zipcodes for California are!

In [ ]:
# 2c YOUR CODE HERE
zip_codes = pd.read_csv('/Users/lacey/github/mids-datasci200-summer2026-lacey-payne/california-zip-codes.csv', index_col=False, header=None, names=['zip'], dtype={'zip': str})
#The .csv file for CA zip codes was downloaded from worldpopulationreview.com. The file was then saved to the same folder as the P00000001-CA.csv file.

df["contbr_zip"] = df["contbr_zip"].astype(str)
df["zip5"] = df["contbr_zip"].str[:5]
ca_zips = zip_codes["zip"].astype(str)   
df = df[df["zip5"].isin(ca_zips)]
print(df.shape)

**2d.** The receipt amount column has negative donations. After talking with your team, a decision was made that the best course of action is to remove these negative values so that the donation count and amount is more accurate. Print/show the shape of the dataframe after the filtering is complete. (5 points)

In [ ]:
# 2d YOUR CODE HERE
df = df[df["contb_receipt_amt"] >= 0]
print(df.shape)

**2e.** From the dataset drop any columns that won't be used in the analysis. Print/show the shape of the dataframe after the dropping is complete. What columns did you drop and why? (5 points)

In [ ]:
# 2e YOUR CODE HERE
df = df.drop(columns=["cmte_id", "contbr_nm", "contbr_city", "contbr_st", "contbr_employer", "contbr_occupation", "receipt_desc", "memo_cd", "memo_text", "form_tp", "file_num"])
print(df.shape)

- **2e answer here:**
The cmte_id, contbr_nm, contbr_city, contbr_st, contbr_employer, contbr_occupation, receipt_desc, memo_cd, memo_text, form_tp, and file_num columns were removed because they contained information unnecessary for the requested zip code and day of the month analyses. 

**2f.** List any assumptions that you made up to this point or if there is any filtering you think is needed and why:

NOTE: You can look to see if there are any duplicate rows also - please write an assumption on what you would do with these!

- **2f answer here:**
- Now that the data quality is fixed, we should have everything we need to complete the analysis and answer the two questions posed.
- The dataset is complete and includes all donations
- The data is accurate
- It is likely most donations will occur around the 1st and/or 15th of the month to accommodate biweekly paychecks.

I'm adding a coding cell to check for duplicate rows. Since a decision was previously made to remove negative donations for improved donation count and amount accuracy, I assume that duplicate values would also need to be removed to improve donation count and amount accuracy, as these would artificially inflate those numbers.

After writing and running the code in the cell below to check specifically for duplicates in the current (filtered) dataframe rather than the entire dataset as I did in 1i., there were none remaining, likely due to upstream filtering resolving any duplication and its source.

In [ ]:
print(df['tran_id'].duplicated().sum())
df.drop_duplicates(subset='tran_id')
print(df.shape)

***
## 3. Answering the questions (20 points)

Now that the data is cleaned and filterd - let's answer the two questions from your boss! That is use the dataframe from 2e above that has done all of the cleaning & filtering steps!

**3a.** Which zipcode had the highest count of contributions and the most dollar amount? (10 points)

In [ ]:
# 3a YOUR CODE HERE
#count occurences of each zip code in the dataframe for zip code with most contributions and see which zip code contributed the most dollars
zip_summary = df.groupby('zip5')['contb_receipt_amt'].agg(['count', 'sum'])

most_contributions_zip = zip_summary['count'].idxmax()
most_dollars_zip = zip_summary['sum'].idxmax()

print("Zip with most contributions:", most_contributions_zip, "-", zip_summary.loc[most_contributions_zip, 'count'])
print("Zip with most dollar amount:", most_dollars_zip, "-", zip_summary.loc[most_dollars_zip, 'sum'])

- **3a answer here:** 
The same zip code had the highest number of contributions and contributed the highest dollar amount. 94110 made 3,799 contributions totaling $284,398.05. This makes sense as 94110 is the zip code for San Francisco, and the Bay Area houses several of the world's largest tech companies. It would stand to reason that these companies would donate to candidates they believe would best suit thier business interests, and individuals residing in this area would have a high disposable income, which they could use to support political candidates and causes of their choosing. 

**3b.** What day(s) of the month do most people donate? (10 points)

In [ ]:
# 3b YOUR CODE HERE
df['day_of_month'] = df['contb_receipt_dt'].dt.day
print(df['day_of_month'].value_counts().sort_values(ascending=False))

- **3b answer here:** 
The 29th of each month had the highest number of donations, followed by the 31st and then the 30th. This aligns with the assumption regarding when most of the donations were likely to occur; although this isn't the 1st, most individuals are paid using direct deposit and many financial institutions release paychecks 2-3 days early. 

## If you have feedback for this homework, please submit it using the link below:

http://goo.gl/forms/74yCiQTf6k